In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
dataset_path = "/content/drive/MyDrive/20_newsgroups/20_newsgroups"

In [6]:
print(os.listdir(dataset_path))

['talk.religion.misc', 'talk.politics.misc', 'talk.politics.mideast', 'talk.politics.guns', 'soc.religion.christian', 'sci.med', 'sci.space', 'sci.electronics', 'sci.crypt', 'rec.autos', 'rec.sport.hockey', 'misc.forsale', 'rec.motorcycles', 'rec.sport.baseball', 'comp.windows.x', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.os.ms-windows.misc', 'comp.graphics', 'alt.atheism']


In [8]:
import os
docs=[]
count=0
for category in os.listdir(dataset_path):
    category_path = os.path.join(dataset_path, category)

    if os.path.isdir(category_path):
        for file in os.listdir(category_path):

            file_path = os.path.join(category_path, file)

            if os.path.isfile(file_path):
                with open(file_path, "r", errors="ignore") as f:
                    docs.append(f.read())

                count += 1

                if count % 2000 == 0:
                    print("Loaded", count, "documents")

print("Total docs:", len(docs))

Loaded 2000 documents
Loaded 4000 documents
Loaded 6000 documents
Loaded 8000 documents
Loaded 10000 documents
Loaded 12000 documents
Loaded 14000 documents
Loaded 16000 documents
Loaded 18000 documents
Loaded 20000 documents
Total docs: 20043


In [26]:
docs = docs_full


In [27]:
import re

headers = [
"Xref","Path","From","Newsgroups","Subject","Summary","Keywords",
"Message-ID","Date","Expires","Followup-To","Distribution",
"Organization","Approved","Supersedes","Lines",
"Archive-name","Alt-atheism-archive-name","Last-modified","Version","X-Newsreader"
]

pattern = r'^(' + '|'.join(headers) + r'):.*$'

def clean_text(text):

    # remove metadata headers
    text = re.sub(pattern, '', text, flags=re.MULTILINE)

    # remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # remove quoted replies
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)

    # remove discussion artifacts
    text = re.sub(r'^(References|Sender|Reply-To):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^In article .* writes:', '', text, flags=re.MULTILINE)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


clean_docs = [clean_text(doc) for doc in docs]

In [28]:
print(len(clean_docs))

20043


In [29]:
for i in range(5):
    print(clean_docs[i][:200])
    print("\n---\n")

So kill a cow, you're a murderer? -jim halat

---

(Frank O'Dwyer) writes: Which goes faster, a bullet or a snail? How come you can answer that when Einstein proved that there isn't an objective frame of reference? I'm sorry, I can't parse "an objecti

---

In "Casper C. Knies" writes: I don't think it's too far off of the mark, either. No one 'deserves' what they get. Punishment is for the justice system (flawed as it may be). This whole 'well it served

---

Frank, I tried to mail this but it bounced. It is fast moving out of t.a scope, but I didn't know if t.a was the only group of the three that you subscribed to. Apologies to regular t.a folks. You mus

---

(John E. King) writes: Looking at [1] we find that during Roman times "Tyre vied with Sidon for first place in the intellectual life of the period"; that Tyre was the seat of a Christian bishop, event

---



In [30]:
!pip install sentence-transformers

In [31]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
embeddings = model.encode(
    clean_docs,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [33]:
print(embeddings.shape)

(20043, 384)


In [34]:
import numpy as np
embeddings = np.array(embeddings).astype("float32")

In [35]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.0 MB/s eta 0:00:00


In [36]:
import faiss
faiss.normalize_L2(embeddings)

In [37]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

In [38]:
index.add(embeddings)

In [39]:
print(index.ntotal)

20043


In [40]:
query = "religion debate"

query_vector = model.encode([query]).astype("float32")
faiss.normalize_L2(query_vector)

distances, indices = index.search(query_vector, 5)

for i in indices[0]:
    print(clean_docs[i][:200])
    print("----")

NNTP-Posting-Host: mead.u.washington.edu Originator: (Sol Lightman) writes: no its not. its due to the fact that there are two issues here: Religion and religion. religion is personal belief system. R
----
Nntp-Posting-Host: charlie I hope I didn't award custody, Rich. I purposely used "handle" in order to avoid doing so - i.e., that happens to be what religions do (of course there are aberrations like 
----
Some variant is quite popular. This, and other arguments, are discussed in John Leslie Mackie's "The Miracle of Theism: arguments for and against the existence of God". Although Mackie ultimately side
----
On 12-Apr-93 in Environmentalism and paganism user Michael writes: And what of those of us who already have answers to their questions without turning to christianity (or, in my case, any religion)? W
----
NNTP-Posting-Host: imager.llnl.gov (John E. King) writes: [ out-of context quotes from Mr. King deleted, along with the context, thoughtfully provided by Mr. Lamb] John: Isn't 

In [41]:
import pickle

data = {
    "documents": clean_docs,
    "embeddings": embeddings
}

with open("embeddings_data.pkl", "wb") as f:
    pickle.dump(data, f)

In [42]:
import faiss
faiss.write_index(index, "faiss_index.bin")

In [43]:
from google.colab import files
files.download("embeddings_data.pkl")
files.download("faiss_index.bin")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>